In [2]:
# Step 1: The notebooks are inspected first so the correct saved outputs can be identified.
# This is done before loading CSV files because many output names are overlapping.

from pathlib import Path
import nbformat
import re

BASE_DIR = Path(".")

notebooks = [
    "R1_Consent_FL_Diabetes-5.ipynb",
    "TemporalTests.ipynb"
]

# Common saved-output patterns are searched inside notebook code cells.
patterns = [
    r"to_csv\(['\"]([^'\"]+)['\"]",
    r"to_json\(['\"]([^'\"]+)['\"]",
    r"savefig\(['\"]([^'\"]+)['\"]",
    r"dump\([^,]+,\s*['\"]([^'\"]+)['\"]",
    r"open\(['\"]([^'\"]+)['\"]",
    r"np\.save\(['\"]([^'\"]+)['\"]",
]

for nb_name in notebooks:
    nb_path = BASE_DIR / nb_name
    
    print("\n" + "="*80)
    print(f"NOTEBOOK: {nb_name}")
    print("="*80)
    
    if not nb_path.exists():
        print("Notebook not found in this folder.")
        continue
    
    nb = nbformat.read(nb_path, as_version=4)
    saved_files = []
    
    for i, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue
        
        code = cell.source
        
        for pattern in patterns:
            matches = re.findall(pattern, code)
            for m in matches:
                saved_files.append((i + 1, m))
    
    if not saved_files:
        print("No saved-output paths were found.")
    else:
        for cell_no, file_path in saved_files:
            print(f"Cell {cell_no}: {file_path}")


NOTEBOOK: R1_Consent_FL_Diabetes-5.ipynb
Notebook not found in this folder.

NOTEBOOK: TemporalTests.ipynb
No saved-output paths were found.


In [3]:
# Step 1B: All notebooks are searched because the exact file names may be different.
# This helps identify the correct experiment notebook before any outputs are reused.

from pathlib import Path
import nbformat
import re

BASE_DIR = Path(".")

# All notebooks in the current folder and subfolders are listed.
notebooks = sorted(BASE_DIR.rglob("*.ipynb"))

print(f"Total notebooks found: {len(notebooks)}\n")

for nb in notebooks:
    print(nb)

Total notebooks found: 8

.ipynb_checkpoints/02_Observation_Group_Validation-checkpoint.ipynb
.ipynb_checkpoints/Dynamic_Diabetes_FL-checkpoint.ipynb
.ipynb_checkpoints/R1_Consent_FL_Diabetes-checkpoint.ipynb
.ipynb_checkpoints/TemporalTests-checkpoint.ipynb
02_Observation_Group_Validation.ipynb
Dynamic_Diabetes_FL.ipynb
R1_Consent_FL_Diabetes.ipynb
TemporalTests.ipynb


In [4]:
# Step 2: Only the correct notebooks are inspected.
# This avoids mixing outputs from older or experimental notebooks.

import nbformat
import re
from pathlib import Path

notebooks = [
    "R1_Consent_FL_Diabetes.ipynb",
    "TemporalTests.ipynb"
]

patterns = {
    "CSV saved": r"\.to_csv\(['\"]([^'\"]+)['\"]",
    "JSON saved": r"\.to_json\(['\"]([^'\"]+)['\"]",
    "Figure saved": r"\.savefig\(['\"]([^'\"]+)['\"]",
    "Pickle/joblib saved": r"(?:joblib\.dump|pickle\.dump|dump)\([^,]+,\s*['\"]([^'\"]+)['\"]",
    "Numpy saved": r"np\.save\(['\"]([^'\"]+)['\"]",
}

for nb_name in notebooks:
    print("\n" + "="*90)
    print(f"NOTEBOOK: {nb_name}")
    print("="*90)
    
    nb = nbformat.read(nb_name, as_version=4)
    saved_outputs = []

    for i, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue
        
        code = cell.source
        
        for label, pattern in patterns.items():
            matches = re.findall(pattern, code)
            for path in matches:
                saved_outputs.append((i + 1, label, path))

    print(f"Saved outputs found: {len(saved_outputs)}\n")

    for cell_no, label, path in saved_outputs:
        print(f"Cell {cell_no:>3} | {label:<18} | {path}")


NOTEBOOK: R1_Consent_FL_Diabetes.ipynb
Saved outputs found: 0


NOTEBOOK: TemporalTests.ipynb
Saved outputs found: 0



In [5]:
# Step 3: The reusable outputs from the correct notebooks are loaded.
# This is done so previous results can be checked before any new model training is considered.

from pathlib import Path
import pandas as pd
import json

BASE_DIR = Path(".")
TABLE_DIR = BASE_DIR / "pipeline_outputs" / "tables"

files_to_check = [
    "feature_group_mapping.csv",
    "exp3_feature_masking_results.csv",
    "exp4_permutation_importance.csv",
    "exp4_shap_importance.csv",
    "exp4_correlation.csv",
    "exp4_mutual_information.csv",
    "exp4_combined_ranking.csv",
    "exp4_progressive_restriction_results.csv",
]

print("Checking reusable files:\n")

for fname in files_to_check:
    path = TABLE_DIR / fname
    
    print("="*80)
    print(fname)
    
    if not path.exists():
        print("❌ File not found")
        continue
    
    df = pd.read_csv(path)
    print(f"✅ Found | shape = {df.shape}")
    print("Columns:", list(df.columns))
    display(df.head())

Checking reusable files:

feature_group_mapping.csv
✅ Found | shape = (168, 3)
Columns: ['group', 'col_index', 'col_name']


,group,col_index,col_name
0,Patient,12,race_Asian
1,Patient,13,race_Caucasian
2,Patient,14,race_Hispanic
3,Patient,15,race_Other
4,Patient,16,race_Unknown


exp3_feature_masking_results.csv
✅ Found | shape = (18, 8)
Columns: ['Group', 'N_cols_masked', 'Pct_cols_masked', 'Model', 'AUC', 'F1', 'Baseline_AUC', 'AUC_drop']


,Group,N_cols_masked,Pct_cols_masked,Model,AUC,F1,Baseline_AUC,AUC_drop
0,Patient,8,4.8,Logistic Regression,0.6450,0.2013,0.6474,-0.0024
1,Patient,8,4.8,XGBoost,0.6545,0.1765,0.6545,0.0000
2,Patient,8,4.8,LightGBM,0.6743,0.2014,0.6757,-0.0014
3,Observation,10,6.0,Logistic Regression,0.5886,0.2010,0.6474,-0.0588
4,Observation,10,6.0,XGBoost,0.5733,0.1116,0.6545,-0.0812


exp4_permutation_importance.csv
✅ Found | shape = (6, 8)
Columns: ['Group', 'LR_drop', 'XGB_drop', 'LGBM_drop', 'Avg_drop', 'LR_std', 'XGB_std', 'LGBM_std']


...

,Group,Avg_MI,Max_MI,Sum_MI
0,Observation,0.002958,0.012438,0.029581
1,Clinical,0.001239,0.003304,0.008671
2,Medication,0.001217,0.004035,0.060843
3,Diagnosis,0.001010,0.004133,0.021203
4,Specialty,0.000884,0.005404,0.063615


exp4_combined_ranking.csv
✅ Found | shape = (6, 12)
Columns: ['Group', 'Perm_drop', 'SHAP_avg', 'Pearson', 'Spearman', 'MI_avg', 'Rank_Perm', 'Rank_SHAP_avg', 'Rank_Pearson', 'Rank_Spearman', 'Rank_MI_avg', 'Avg_rank']


,Group,Perm_drop,SHAP_avg,Pearson,Spearman,MI_avg,Rank_Perm,Rank_SHAP_avg,Rank_Pearson,Rank_Spearman,Rank_MI_avg,Avg_rank
0,Observation,-0.122332,0.110717,0.046744,0.047715,0.002958,1,1,1,1,1,1.0
1,Clinical,-0.002894,0.021618,0.020434,0.020483,0.001239,5,2,2,2,2,2.6
2,Diagnosis,-0.005500,0.017143,0.010950,0.010950,0.001010,4,3,4,4,4,3.8
3,Patient,-0.002659,0.016445,0.012737,0.012976,0.000518,6,4,3,3,6,4.4
4,Medication,-0.007057,0.006587,0.007723,0.007723,0.001217,2,5,6,6,3,4.4


exp4_progressive_restriction_results.csv
✅ Found | shape = (21, 11)
Columns: ['Step', 'Groups_masked', 'N_cols_masked', 'Pct_cols_masked', 'Model', 'AUC', 'F1', 'Baseline_AUC', 'AUC_drop', 'Group_added', 'Time_s']


,Step,Groups_masked,N_cols_masked,Pct_cols_masked,Model,AUC,F1,Baseline_AUC,AUC_drop,Group_added,Time_s
0,0,NaN,0,0.0,Logistic Regression,0.6474,0.2015,0.6757,0.0000,NaN,NaN
1,0,NaN,0,0.0,XGBoost,0.6545,0.1744,0.6474,0.0000,NaN,NaN
2,0,NaN,0,0.0,LightGBM,0.6757,0.2014,0.6545,0.0000,NaN,NaN
3,1,Diagnosis,21,12.5,Logistic Regression,0.6482,0.2015,0.6757,-0.0275,Diagnosis,584.7
4,1,Diagnosis,21,12.5,XGBoost,0.6593,0.1935,0.6474,0.0119,Diagnosis,34.1


In [6]:
# Step 4: Observation variables are extracted from the saved feature mapping.
# This is done to understand what the Observation group actually contains.
# Conor specifically asked for this before any further experiments.

import pandas as pd

mapping = pd.read_csv(
    "pipeline_outputs/tables/feature_group_mapping.csv"
)

obs_vars = mapping[mapping["group"] == "Observation"]

print(f"\nObservation variables: {len(obs_vars)}\n")

display(obs_vars[["col_index", "col_name"]].sort_values("col_index"))


Observation variables: 10



,col_index,col_name
15,1,admission_type_id
16,2,discharge_disposition_id
17,3,admission_source_id
8,4,time_in_hospital
9,5,num_lab_procedures
10,6,num_procedures
11,7,num_medications
12,8,number_outpatient
13,9,number_emergency
14,10,number_inpatient


In [7]:
# Step 5: Observation variables are ranked by simple association with the target.
# This is done to identify whether one variable dominates the Observation group.
# Conor specifically raised the possibility that only one or two variables may be driving the result.

import pandas as pd
from sklearn.feature_selection import mutual_info_classif

# Load train data used in experiments
X_train = pd.read_csv("pipeline_outputs/tables/X_train_scaled.csv")
y_train = pd.read_csv("pipeline_outputs/tables/y_train.csv").squeeze()

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

# Mutual information is calculated for Observation variables only.
mi = mutual_info_classif(
    X_train[obs_vars],
    y_train,
    random_state=42
)

mi_df = pd.DataFrame({
    "Variable": obs_vars,
    "Mutual_Information": mi
}).sort_values(
    "Mutual_Information",
    ascending=False
)

display(mi_df)

,Variable,Mutual_Information
9,number_inpatient,0.016248
1,discharge_disposition_id,0.008690
8,number_emergency,0.007718
7,number_outpatient,0.004573
0,admission_type_id,0.004101
5,num_procedures,0.002545
2,admission_source_id,0.002101
3,time_in_hospital,0.002071
6,num_medications,0.000514
4,num_lab_procedures,0.000000


In [8]:
# Step 6: Observation variables are checked individually against the target.
# This is done to identify whether any variable shows suspiciously strong association.
# A leakage pattern would normally appear as one variable dominating all others.

import pandas as pd

X_train = pd.read_csv("pipeline_outputs/tables/X_train_scaled.csv")
y_train = pd.read_csv("pipeline_outputs/tables/y_train.csv").squeeze()

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

rows = []

for col in obs_vars:
    
    pearson = X_train[col].corr(y_train)
    
    rows.append({
        "Variable": col,
        "Abs_Pearson": abs(pearson),
        "Pearson": pearson
    })

corr_df = pd.DataFrame(rows)

corr_df = corr_df.sort_values(
    "Abs_Pearson",
    ascending=False
)

display(corr_df)

,Variable,Abs_Pearson,Pearson
9,number_inpatient,0.165284,0.165284
8,number_emergency,0.057976,0.057976
1,discharge_disposition_id,0.051084,0.051084
3,time_in_hospital,0.043468,0.043468
6,num_medications,0.038204,0.038204
4,num_lab_procedures,0.019755,0.019755
7,number_outpatient,0.018023,0.018023
5,num_procedures,0.009343,-0.009343
0,admission_type_id,0.009265,-0.009265
2,admission_source_id,0.006820,0.006820


Observation dominance does not appear to arise from a single highly predictive variable. The strongest individual association was number_inpatient (r=0.165), followed by number_emergency (r=0.058) and discharge_disposition_id (r=0.051). All remaining variables showed weak associations with readmission, suggesting that predictive signal is distributed across multiple healthcare utilisation indicators rather than driven by a single leakage-prone feature.

In [9]:
# Step 7: Observation variables are ranked by removal impact.
# This is done to determine whether Observation dominance is caused
# by one variable or by the combined contribution of several variables.

import pandas as pd

X_train = pd.read_csv("pipeline_outputs/tables/X_train_scaled.csv")
X_test  = pd.read_csv("pipeline_outputs/tables/X_test_scaled_168.csv")

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

missing_train = [c for c in obs_vars if c not in X_train.columns]
missing_test  = [c for c in obs_vars if c not in X_test.columns]

print("\nMissing in train:", missing_train)
print("Missing in test :", missing_test)

Train shape: (81412, 2400)
Test shape : (20354, 168)

Missing in train: []
Missing in test : []


In [10]:
# Step 8: Candidate train datasets are inspected.
# This is done to identify the exact train matrix used in the feature-group experiments.

import pandas as pd

candidate_files = [
    "pipeline_outputs/tables/X_train_scaled.csv",
    "pipeline_outputs/tables/X_test_scaled_168.csv",
    "paper_outputs/step4D_X_train.csv",
    "paper_outputs/step4D_X_test.csv",
]

for f in candidate_files:
    
    try:
        df = pd.read_csv(f, nrows=5)
        print("\n" + "="*80)
        print(f)
        print("Shape (5 rows preview):", df.shape)
        print("First 15 columns:")
        print(list(df.columns[:15]))
        
    except Exception as e:
        print(f"\n{f}")
        print("ERROR:", e)


pipeline_outputs/tables/X_train_scaled.csv
Shape (5 rows preview): (5, 2400)
First 15 columns:
['age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'race_Asian', 'race_Caucasian', 'race_Hispanic']

pipeline_outputs/tables/X_test_scaled_168.csv
Shape (5 rows preview): (5, 168)
First 15 columns:
['age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'race_Asian', 'race_Caucasian', 'race_Hispanic']

paper_outputs/step4D_X_train.csv
Shape (5 rows preview): (5, 165)
First 15 columns:
['age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'nu

Evidence against leakage
Observation ranked first by:
masking
permutation importance
SHAP
correlation
mutual information
Observation variables are clinically meaningful:
admission characteristics
healthcare utilisation
prior admissions
emergency visits
length of stay
No single variable appears suspicious:
highest MI = 0.016 (number_inpatient)
highest correlation = 0.165 (number_inpatient)

These are moderate associations, not leakage.

In [11]:
# Step 9: Each Observation variable is evaluated individually.
# This is done to determine whether one variable can predict readmission on its own.
# If no variable performs strongly alone, Observation importance is likely distributed.

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_train = pd.read_csv("paper_outputs/step4D_X_train.csv")
X_test  = pd.read_csv("paper_outputs/step4D_X_test.csv")

y_train = pd.read_csv("paper_outputs/step4D_y_train.csv").squeeze()
y_test  = pd.read_csv("paper_outputs/step4D_y_test.csv").squeeze()

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

results = []

for var in obs_vars:

    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    model.fit(
        X_train[[var]],
        y_train
    )

    probs = model.predict_proba(
        X_test[[var]]
    )[:,1]

    auc = roc_auc_score(
        y_test,
        probs
    )

    results.append({
        "Variable": var,
        "Single_Variable_AUC": round(auc, 4)
    })

results = pd.DataFrame(results)

results = results.sort_values(
    "Single_Variable_AUC",
    ascending=False
)

display(results)

,Variable,Single_Variable_AUC
9,number_inpatient,0.6078
1,discharge_disposition_id,0.5522
3,time_in_hospital,0.5482
6,num_medications,0.5450
8,number_emergency,0.5341
4,num_lab_procedures,0.5214
0,admission_type_id,0.5192
7,number_outpatient,0.5189
5,num_procedures,0.5137
2,admission_source_id,0.5072


In [12]:
# Step 10: Each Observation variable is tested as a single predictor across the three main models.
# This is done to check whether any one Observation variable alone can explain the Observation effect.
# A leakage-like variable would show unusually high AUC across models.

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Data used in the grouped-feature experiments is loaded.
X_train = pd.read_csv("paper_outputs/step4D_X_train.csv")
X_test  = pd.read_csv("paper_outputs/step4D_X_test.csv")

y_train = pd.read_csv("paper_outputs/step4D_y_train.csv").squeeze()
y_test  = pd.read_csv("paper_outputs/step4D_y_test.csv").squeeze()

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

models = {
    "LR": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),
    
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
}

rows = []

for var in obs_vars:
    
    for model_name, model in models.items():
        
        # The model is trained using one Observation variable only.
        # This checks whether that variable alone has suspiciously strong predictive power.
        model.fit(
            X_train[[var]],
            y_train
        )
        
        probs = model.predict_proba(
            X_test[[var]]
        )[:, 1]
        
        auc = roc_auc_score(
            y_test,
            probs
        )
        
        rows.append({
            "Variable": var,
            "Model": model_name,
            "Single_Variable_AUC": round(auc, 4)
        })

single_var_auc = pd.DataFrame(rows)

# Mean AUC across models is added for easier ranking.
summary = (
    single_var_auc
    .groupby("Variable")["Single_Variable_AUC"]
    .agg(["mean", "min", "max"])
    .reset_index()
    .rename(columns={
        "mean": "Mean_AUC",
        "min": "Min_AUC",
        "max": "Max_AUC"
    })
)

summary["Mean_AUC"] = summary["Mean_AUC"].round(4)
summary["Min_AUC"] = summary["Min_AUC"].round(4)
summary["Max_AUC"] = summary["Max_AUC"].round(4)

summary = summary.sort_values(
    "Mean_AUC",
    ascending=False
)

display(single_var_auc)
display(summary)

# Results are saved for the observation-validation notebook/paper table.
single_var_auc.to_csv(
    "observation_single_variable_auc_by_model.csv",
    index=False
)

summary.to_csv(
    "observation_single_variable_auc_summary.csv",
    index=False
)

,Variable,Model,Single_Variable_AUC
0,admission_type_id,LR,0.5192
1,admission_type_id,XGBoost,0.5176
2,admission_type_id,LightGBM,0.5176
3,discharge_disposition_id,LR,0.5522
4,discharge_disposition_id,XGBoost,0.5887
5,discharge_disposition_id,LightGBM,0.5888
6,admission_source_id,LR,0.5072
7,admission_source_id,XGBoost,0.5167
8,admission_source_id,LightGBM,0.5169
9,time_in_hospital,LR,0.5482


,Variable,Mean_AUC,Min_AUC,Max_AUC
7,number_inpatient,0.6078,0.6078,0.6078
2,discharge_disposition_id,0.5766,0.5522,0.5888
9,time_in_hospital,0.5495,0.5482,0.5501
4,num_medications,0.5461,0.5450,0.5468
6,number_emergency,0.5341,0.5341,0.5341
3,num_lab_procedures,0.5215,0.5207,0.5223
8,number_outpatient,0.5193,0.5189,0.5196
1,admission_type_id,0.5181,0.5176,0.5192
5,num_procedures,0.5175,0.5137,0.5194
0,admission_source_id,0.5136,0.5072,0.5169


### Observation-only versus Non-Observation-only predictive comparison

The purpose of this analysis is to test whether the predictive signal is mainly concentrated in the Observation group or distributed across the remaining consent groups. Earlier checks showed that no single Observation variable has unusually high correlation, mutual information, or standalone AUC. Therefore, the next logical step is to evaluate the Observation group as a combined clinical-utilisation construct.

Three feature settings are compared across the main model families: Observation-only, Non-Observation-only, and Full feature set. If Observation-only achieves performance close to the full model, while Non-Observation-only performs substantially lower, this would support the interpretation that Observation variables carry the core readmission signal. If Non-Observation-only performs similarly to the full model, then the Observation dominance would be less convincing.

In [13]:
# Step 11: Observation-only, Non-Observation-only, and Full models are compared.
# This is done to test whether the Observation group carries the main predictive signal as a group.
# The comparison is repeated across LR, XGBoost, and LightGBM for stronger evidence.

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# The grouped-feature train/test data used in the main experiments is loaded.
X_train = pd.read_csv("paper_outputs/step4D_X_train.csv")
X_test  = pd.read_csv("paper_outputs/step4D_X_test.csv")

y_train = pd.read_csv("paper_outputs/step4D_y_train.csv").squeeze()
y_test  = pd.read_csv("paper_outputs/step4D_y_test.csv").squeeze()

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

non_obs_vars = [
    c for c in X_train.columns
    if c not in obs_vars
]

feature_sets = {
    "Observation only": obs_vars,
    "Non-Observation only": non_obs_vars,
    "Full feature set": list(X_train.columns)
}

models = {
    "LR": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),
    
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
}

rows = []

for feature_set_name, cols in feature_sets.items():
    
    for model_name, model in models.items():
        
        # The model is trained using one feature setting at a time.
        # This isolates the predictive contribution of Observation versus all other groups.
        model.fit(
            X_train[cols],
            y_train
        )
        
        probs = model.predict_proba(
            X_test[cols]
        )[:, 1]
        
        preds = (probs >= 0.5).astype(int)
        
        rows.append({
            "Feature_Set": feature_set_name,
            "Model": model_name,
            "N_Features": len(cols),
            "AUC": round(roc_auc_score(y_test, probs), 4),
            "Average_Precision": round(average_precision_score(y_test, probs), 4),
            "F1_at_0.5": round(f1_score(y_test, preds), 4)
        })

obs_vs_nonobs_results = pd.DataFrame(rows)

display(obs_vs_nonobs_results)

# A pivot table is created for easier comparison in the paper.
auc_table = obs_vs_nonobs_results.pivot(
    index="Model",
    columns="Feature_Set",
    values="AUC"
).reset_index()

display(auc_table)

# Results are saved for later use in the paper.
obs_vs_nonobs_results.to_csv(
    "observation_vs_nonobservation_model_results.csv",
    index=False
)

auc_table.to_csv(
    "observation_vs_nonobservation_auc_table.csv",
    index=False
)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Feature_Set,Model,N_Features,AUC,Average_Precision,F1_at_0.5
0,Observation only,LR,10,0.6387,0.1959,0.0282
1,Observation only,XGBoost,10,0.6721,0.2219,0.0096
2,Observation only,LightGBM,10,0.6718,0.2210,0.0096
3,Non-Observation only,LR,155,0.5880,0.1470,0.0000
4,Non-Observation only,XGBoost,155,0.5889,0.1479,0.0000
5,Non-Observation only,LightGBM,155,0.5889,0.1476,0.0000
6,Full feature set,LR,165,0.6480,0.2015,0.0249
7,Full feature set,XGBoost,165,0.6746,0.2230,0.0070
8,Full feature set,LightGBM,165,0.6745,0.2235,0.0070


Feature_Set,Model,Full feature set,Non-Observation only,Observation only
0,LR,0.6480,0.5880,0.6387
1,LightGBM,0.6745,0.5889,0.6718
2,XGBoost,0.6746,0.5889,0.6721


### Leave-one-variable-out ablation within the Observation group

The previous analyses established that the Observation group contains most of the predictive signal and that no single Observation variable exhibits unusually strong standalone predictive power. The next step is to determine whether the Observation effect is driven by a single variable or arises from the combined contribution of multiple healthcare utilisation indicators.

A leave-one-variable-out ablation analysis is performed by removing one Observation variable at a time while retaining all remaining features. The resulting performance is compared with the full model. Larger performance drops indicate greater dependence on that variable. If performance decreases are distributed across several variables, this would support the interpretation that Observation dominance emerges from the combined contribution of multiple utilisation-related features rather than a single dominant predictor.

In [14]:
# Step 12: Observation-variable ablation analysis is performed.
# One Observation variable is removed at a time while all other features are retained.
# This is done to identify whether Observation dominance is driven by one variable
# or emerges from the combined contribution of multiple healthcare utilisation indicators.

import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

# Data used in the main grouped-feature experiments is loaded.
X_train = pd.read_csv("paper_outputs/step4D_X_train.csv")
X_test  = pd.read_csv("paper_outputs/step4D_X_test.csv")

y_train = pd.read_csv("paper_outputs/step4D_y_train.csv").squeeze()
y_test  = pd.read_csv("paper_outputs/step4D_y_test.csv").squeeze()

obs_vars = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

# Baseline model is trained using all features.
baseline_model = LGBMClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

baseline_model.fit(X_train, y_train)

baseline_probs = baseline_model.predict_proba(X_test)[:,1]

baseline_auc = roc_auc_score(
    y_test,
    baseline_probs
)

print(f"Baseline AUC = {baseline_auc:.4f}")

rows = []

for var in obs_vars:

    remaining_cols = [
        c for c in X_train.columns
        if c != var
    ]

    model = LGBMClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    model.fit(
        X_train[remaining_cols],
        y_train
    )

    probs = model.predict_proba(
        X_test[remaining_cols]
    )[:,1]

    auc = roc_auc_score(
        y_test,
        probs
    )

    rows.append({
        "Removed_Variable": var,
        "AUC": round(auc,4),
        "AUC_Drop": round(
            baseline_auc - auc,
            4
        )
    })

ablation_results = pd.DataFrame(rows)

ablation_results = ablation_results.sort_values(
    "AUC_Drop",
    ascending=False
)

display(ablation_results)

ablation_results.to_csv(
    "observation_variable_ablation_lgbm.csv",
    index=False
)

Baseline AUC = 0.6745


,Removed_Variable,AUC,AUC_Drop
9,number_inpatient,0.6443,0.0302
1,discharge_disposition_id,0.6475,0.0269
5,num_procedures,0.6739,0.0006
8,number_emergency,0.6744,0.0001
7,number_outpatient,0.6747,-0.0002
4,num_lab_procedures,0.6748,-0.0003
0,admission_type_id,0.6752,-0.0007
3,time_in_hospital,0.6754,-0.0009
2,admission_source_id,0.6756,-0.0011
6,num_medications,0.6756,-0.0011


### Decomposing the Observation Group

Previous analyses established that the Observation group contains most of the predictive signal. Leave-one-variable-out ablation further showed that performance is primarily affected by the removal of number_inpatient and discharge_disposition_id, while the remaining Observation variables have negligible impact when evaluated individually.

The purpose of this analysis is to determine whether Observation dominance is effectively explained by these two variables alone or whether the remaining Observation variables still contribute meaningful predictive information when considered collectively.

Four feature settings are compared:

1. Full model (all features)
2. Full Observation group (10 variables)
3. Top-2 Observation variables only
4. Remaining Observation variables only

This analysis helps distinguish between a two-variable explanation and a broader healthcare-utilisation explanation of the Observation effect.
| Feature Set             | LR    | XGB   | LGBM  |
| ----------------------- | ----- | ----- | ----- |
| Full Model              | 0.648 | 0.675 | 0.675 |
| All Observation         | 0.639 | 0.672 | 0.672 |
| Top 2 Observation       | ?     | ?     | ?     |
| Remaining 8 Observation | ?     | ?     | ?     |


### Observation Group Decomposition Across Model Families

This analysis tests whether the Observation group effect is mainly explained by the two strongest variables or by the broader set of Observation variables. The comparison is repeated across LR, XGBoost, and LightGBM so the conclusion is not dependent on one model family.

Four feature settings are compared: Full model, All Observation variables, Top-2 Observation variables, and the Remaining 8 Observation variables.

In [15]:
# Step 13: Observation-group decomposition is repeated across the three main models.
# This is done to check whether the Top-2 Observation explanation is consistent across model families.

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

X_train = pd.read_csv("paper_outputs/step4D_X_train.csv")
X_test  = pd.read_csv("paper_outputs/step4D_X_test.csv")

y_train = pd.read_csv("paper_outputs/step4D_y_train.csv").squeeze()
y_test  = pd.read_csv("paper_outputs/step4D_y_test.csv").squeeze()

all_obs = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

top2_obs = [
    "number_inpatient",
    "discharge_disposition_id"
]

remaining_obs = [
    c for c in all_obs
    if c not in top2_obs
]

feature_sets = {
    "Full model": list(X_train.columns),
    "All Observation": all_obs,
    "Top-2 Observation": top2_obs,
    "Remaining 8 Observation": remaining_obs
}

models = {
    "LR": LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),
    
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
}

rows = []

for feature_set_name, cols in feature_sets.items():
    
    for model_name, model in models.items():
        
        # Each model is trained using one feature set at a time.
        # This isolates whether performance comes from all Observation variables,
        # only the Top-2 variables, or the remaining Observation variables.
        model.fit(
            X_train[cols],
            y_train
        )
        
        probs = model.predict_proba(
            X_test[cols]
        )[:, 1]
        
        rows.append({
            "Feature_Set": feature_set_name,
            "Model": model_name,
            "N_Features": len(cols),
            "AUC": round(roc_auc_score(y_test, probs), 4),
            "Average_Precision": round(average_precision_score(y_test, probs), 4)
        })

decomposition_results = pd.DataFrame(rows)

display(decomposition_results)

# AUC is reshaped into a paper-friendly table.
decomposition_auc_table = decomposition_results.pivot(
    index="Feature_Set",
    columns="Model",
    values="AUC"
).reset_index()

# Feature-set order is fixed for easier reading.
feature_order = [
    "Full model",
    "All Observation",
    "Top-2 Observation",
    "Remaining 8 Observation"
]

decomposition_auc_table["Feature_Set"] = pd.Categorical(
    decomposition_auc_table["Feature_Set"],
    categories=feature_order,
    ordered=True
)

decomposition_auc_table = decomposition_auc_table.sort_values("Feature_Set")

display(decomposition_auc_table)

decomposition_results.to_csv(
    "observation_group_decomposition_all_models.csv",
    index=False
)

decomposition_auc_table.to_csv(
    "observation_group_decomposition_auc_table.csv",
    index=False
)

,Feature_Set,Model,N_Features,AUC,Average_Precision
0,Full model,LR,165,0.6481,0.2016
1,Full model,XGBoost,165,0.6746,0.2230
2,Full model,LightGBM,165,0.6745,0.2235
3,All Observation,LR,10,0.6387,0.1959
4,All Observation,XGBoost,10,0.6721,0.2219
5,All Observation,LightGBM,10,0.6718,0.2210
6,Top-2 Observation,LR,2,0.6341,0.1878
7,Top-2 Observation,XGBoost,2,0.6655,0.2072
8,Top-2 Observation,LightGBM,2,0.6659,0.2079
9,Remaining 8 Observation,LR,8,0.5850,0.1480


Model,Feature_Set,LR,LightGBM,XGBoost
1,Full model,0.6481,0.6745,0.6746
0,All Observation,0.6387,0.6718,0.6721
3,Top-2 Observation,0.6341,0.6659,0.6655
2,Remaining 8 Observation,0.5850,0.5903,0.5906


## Observation Group Validation and Decomposition

### Clinical Interpretation of Observation Variables

The Observation group consists of ten variables that primarily capture healthcare utilisation, admission characteristics, treatment intensity, and prior interactions with the healthcare system.

| Variable | Clinical Interpretation |
|-----------|------------------------|
| admission_type_id | Type of hospital admission |
| discharge_disposition_id | Discharge destination after hospitalisation |
| admission_source_id | Source of admission |
| time_in_hospital | Length of hospital stay |
| num_lab_procedures | Number of laboratory investigations |
| num_procedures | Number of procedures performed |
| num_medications | Medication burden and treatment intensity |
| number_outpatient | Prior outpatient utilisation |
| number_emergency | Prior emergency department utilisation |
| number_inpatient | Prior inpatient utilisation |

Collectively, these variables represent healthcare utilisation and disease burden rather than diagnosis-specific information.

---

### Validation of Observation Group Dominance

Multiple independent analyses consistently identified the Observation group as the most important feature category.

| Analysis | Key Result |
|-----------|-----------|
| Feature masking | Largest performance degradation when Observation variables were removed |
| Permutation importance | Observation ranked first across all models |
| SHAP analysis | Observation ranked first across all models |
| Correlation analysis | Observation exhibited the strongest association with readmission |
| Mutual information analysis | Observation contained the highest information content relative to readmission |

These findings consistently indicate that the Observation group contains substantially more predictive information than any other consent category.

---

### Leakage Assessment

To determine whether Observation dominance could be explained by target leakage, correlation, mutual information, and single-variable predictive analyses were performed.

#### Correlation Analysis

| Variable | Absolute Correlation |
|-----------|-----------|
| number_inpatient | 0.165 |
| number_emergency | 0.058 |
| discharge_disposition_id | 0.051 |

All remaining Observation variables exhibited weak correlations with readmission. No variable demonstrated the exceptionally high correlations typically associated with leakage.

#### Mutual Information Analysis

| Variable | Mutual Information |
|-----------|-----------|
| number_inpatient | 0.016 |
| discharge_disposition_id | 0.009 |
| number_emergency | 0.008 |

Mutual information values were modest and consistent with clinically meaningful predictors rather than variables that directly encode the target.

---

### Single-Variable Predictive Power

Each Observation variable was evaluated independently using Logistic Regression, XGBoost, and LightGBM.

| Variable | Mean AUC |
|-----------|-----------|
| number_inpatient | 0.608 |
| discharge_disposition_id | 0.577 |
| time_in_hospital | 0.550 |
| num_medications | 0.546 |
| number_emergency | 0.534 |

No individual Observation variable achieved high predictive performance in isolation. The strongest variable, number_inpatient, achieved a mean AUC of 0.608, indicating that Observation dominance cannot be attributed to a single highly predictive feature.

---

### Observation-Only versus Non-Observation Models

To quantify the predictive contribution of the Observation group, models were trained using Observation variables only, Non-Observation variables only, and the complete feature set.

| Model | Full Model | Observation Only | Non-Observation Only |
|---------|---------|---------|---------|
| Logistic Regression | 0.648 | 0.639 | 0.588 |
| XGBoost | 0.675 | 0.672 | 0.589 |
| LightGBM | 0.675 | 0.672 | 0.589 |

Observation-only models retained nearly all predictive performance achieved by the complete feature set, while the remaining 155 features produced substantially lower performance. This indicates that the majority of predictive signal is concentrated within the Observation group.

---

### Observation Variable Ablation

A leave-one-variable-out ablation analysis was performed within the Observation group.

| Removed Variable | AUC Drop |
|-----------|-----------|
| number_inpatient | 0.030 |
| discharge_disposition_id | 0.027 |
| num_procedures | 0.001 |
| All remaining variables | ≈ 0 |

Only two variables produced meaningful performance degradation when removed: number_inpatient and discharge_disposition_id. Removal of all other Observation variables had negligible impact on model performance.

---

### Observation Group Decomposition

To determine whether Observation dominance was driven primarily by these two variables, the Observation group was decomposed into its strongest and weakest components.

| Feature Set | LR | XGBoost | LightGBM |
|------------|------|------|------|
| Full Model | 0.648 | 0.675 | 0.675 |
| All Observation Variables | 0.639 | 0.672 | 0.672 |
| Top-2 Observation Variables | 0.634 | 0.666 | 0.666 |
| Remaining 8 Observation Variables | 0.585 | 0.591 | 0.590 |

The two-variable model recovered almost all predictive performance achieved by the complete Observation group. In contrast, the remaining eight Observation variables performed substantially worse and contributed relatively little predictive value when evaluated collectively.

---

### Summary of Findings

1. Observation dominance is consistently supported across masking, permutation importance, SHAP, correlation, and mutual information analyses.

2. Observation variables represent clinically meaningful healthcare utilisation indicators rather than diagnosis-specific information.

3. No individual Observation variable exhibits characteristics suggestive of target leakage.

4. Observation-only models recover nearly all predictive performance achieved by full-feature models.

5. Most of the predictive value contained within the Observation group is concentrated in two variables: previous inpatient utilisation (`number_inpatient`) and discharge disposition (`discharge_disposition_id`).

6. The remaining Observation variables contribute comparatively little incremental predictive information once these two variables are available.

---

### Interpretation

The apparent dominance of the Observation consent group is not attributable to target leakage or a single highly predictive feature. Instead, predictive performance is largely driven by two clinically meaningful healthcare utilisation indicators—previous inpatient utilisation and discharge disposition—which together recover nearly all of the predictive value contained within the broader Observation group. Observation-only models achieve performance comparable to full-feature models, while non-Observation features contribute relatively little additional predictive information. These findings suggest that the impact of consent withdrawal depends strongly on the specific clinical information being removed rather than simply the number of features affected.